In [5]:
# ═══════════════════════════════════════════════════
# CELL 1 — CONFIG
# ═══════════════════════════════════════════════════

# ── Corpus ────────────────────────────────────────
CORPUS_URL  = "http://mattmahoney.net/dc/text8.zip"
CORPUS_PATH = "data/text8"
MAX_TOKENS  = None

# ── Vocabulary ────────────────────────────────────
VOCAB_SIZE  = None
MIN_COUNT   = 10

# ── Embedding ─────────────────────────────────────
EMBED_DIM   = 300
MRL_NESTING = [50, 100, 150, 200, 250, 300]

# ── Training ──────────────────────────────────────
WINDOW_SIZE = 5
NEG_SAMPLES = 10
EPOCHS      = 10          # per call to phase_train; resumes automatically
BATCH_SIZE  = 65536       # split across 2 GPUs → 32768 each
LR          = 0.001
SUBSAMPLE_T = 1e-5

# ── Paths ─────────────────────────────────────────
SAVE_DIR    = "checkpoints"
STD_CKPT    = f"{SAVE_DIR}/standard_w2v.pt"
MRL_CKPT    = f"{SAVE_DIR}/mrl_w2v.pt"
RESULTS_DIR = "results"
DEVICE      = "cuda"

print("Config loaded.")


Config loaded.


In [6]:
# ═══════════════════════════════════════════════════
# CELL 2 — DATA
# v2 fixes vs v1:
#   ▸ Vectorised pair building (numpy, not Python loop)
#     → ~20× faster pair construction
#   ▸ int32 pairs → half the RAM of int64
#   ▸ memmap cache → second run loads in <1s
# ═══════════════════════════════════════════════════

import os, zipfile, urllib.request, pickle
from collections import Counter
import numpy as np
import torch
from torch.utils.data import Dataset

PAIRS_CACHE = "data/skipgram_pairs.npy"


def download_corpus():
    os.makedirs("data", exist_ok=True)
    if not os.path.exists(CORPUS_PATH):
        zip_path = "data/text8.zip"
        if not os.path.exists(zip_path):
            print("Downloading text8 corpus (~100MB) …")
            urllib.request.urlretrieve(CORPUS_URL, zip_path)
        print("Extracting …")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall("data/")
    print(f"Corpus ready: {CORPUS_PATH}")


def load_tokens(max_tokens=MAX_TOKENS):
    with open(CORPUS_PATH, "r") as f:
        text = f.read()          # read entire file when max_tokens is None
    tokens = text.split()
    return tokens if max_tokens is None else tokens[:max_tokens]


class Vocabulary:
    def __init__(self, tokens):
        counts = Counter(tokens)
        vocab  = [w for w, c in counts.most_common(VOCAB_SIZE) if c >= MIN_COUNT]
        self.w2i    = {w: i for i, w in enumerate(vocab)}
        self.i2w    = vocab
        self.size   = len(vocab)
        self.counts = np.array([counts[w] for w in vocab], dtype=np.float32)

        freq = self.counts / self.counts.sum()
        self.keep_prob = np.minimum(
            1.0,
            np.sqrt(SUBSAMPLE_T / np.maximum(freq, 1e-12))
            + SUBSAMPLE_T / np.maximum(freq, 1e-12),
        ).astype(np.float32)

    def encode(self, tokens):
        return np.array([self.w2i[t] for t in tokens if t in self.w2i], dtype=np.int32)

    def neg_sample_probs(self):
        p = self.counts ** 0.75
        return (p / p.sum()).astype(np.float32)


class SkipGramDataset(Dataset):
    def __init__(self, vocab, tokens):
        if os.path.exists(PAIRS_CACHE):
            print(f"Loading cached pairs from {PAIRS_CACHE} …")
            self.pairs = np.load(PAIRS_CACHE, mmap_mode="r")
        else:
            print("Building skip-gram pairs (vectorised) …")
            ids  = vocab.encode(tokens)
            mask = np.random.rand(len(ids)).astype(np.float32) < vocab.keep_prob[ids]
            ids  = ids[mask]

            # ── Vectorised window expansion — replaces slow Python loop ────────
            # For each offset 1..W, create (center, context) pairs by array slicing
            # This is ~20× faster than the explicit loop used in v1
            chunk_list = []
            for offset in range(1, WINDOW_SIZE + 1):
                # center→context and context→center (both directions)
                fwd = np.stack([ids[:-offset], ids[offset:]], axis=1)
                bwd = np.stack([ids[offset:],  ids[:-offset]], axis=1)
                chunk_list.extend([fwd, bwd])
            self.pairs = np.concatenate(chunk_list, axis=0).astype(np.int32)
            # ────────────────────────────────────────────────────────────────────

            np.save(PAIRS_CACHE, self.pairs)
            print(f"Cached pairs to {PAIRS_CACHE}")

        print(f"Dataset: {len(self.pairs):,} pairs  ({self.pairs.nbytes/1e6:.0f} MB, int32)")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return torch.tensor(self.pairs[idx], dtype=torch.long)


print("Data module loaded.")


Data module loaded.


In [7]:
# ═══════════════════════════════════════════════════
# CELL 3 — MODELS
# v2 fixes vs v1:
#   ▸ Dense embeddings only (no sparse=True)
#     sparse=True is incompatible with DataParallel,
#     AMP, clip_grad, and torch.compile on CUDA
#   ▸ Fused MRL loop: W_in/W_out looked up ONCE
#     at full 300d; prefix slices are memory views
# ═══════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


def neg_sampling_loss(center, context, negatives):
    """
    L = -log σ(v_c · v_+) - Σ log σ(-v_c · v_k)
    center:    (B, d)
    context:   (B, d)
    negatives: (B, K, d)
    """
    pos_loss = F.logsigmoid((center * context).sum(dim=1))
    neg_loss = F.logsigmoid(
        -torch.bmm(negatives, center.unsqueeze(2)).squeeze(2)
    ).sum(dim=1)
    return -(pos_loss + neg_loss).mean()


class _Base(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # DENSE embeddings — required for DataParallel + CUDA correctness
        self.W_in  = nn.Embedding(vocab_size, EMBED_DIM)
        self.W_out = nn.Embedding(vocab_size, EMBED_DIM)
        nn.init.uniform_(self.W_in.weight,  -0.5/EMBED_DIM, 0.5/EMBED_DIM)
        nn.init.zeros_(self.W_out.weight)

    def get_embeddings(self):
        return self.W_in.weight.detach().cpu().float().numpy()

    def get_prefix(self, m):
        return self.W_in.weight[:, :m].detach().cpu().float().numpy()


class StandardWord2Vec(_Base):
    def forward(self, centers, contexts, negatives):
        return neg_sampling_loss(
            self.W_in(centers),
            self.W_out(contexts),
            self.W_out(negatives),
        )


class MRLWord2Vec(_Base):
    """
    L_MRL = (1/|M|) Σ_{m∈M} L_NS(f(w)[:m], f(c)[:m])

    Key: W_in and W_out looked up once at full D dims.
    Prefix slices [:m] are tensor views — no extra memory.
    """
    def __init__(self, vocab_size, nesting=None):
        super().__init__(vocab_size)
        self.nesting = nesting or MRL_NESTING
        assert self.nesting[-1] == EMBED_DIM
        self.lam = 1.0 / len(self.nesting)

    def forward(self, centers, contexts, negatives):
        vc  = self.W_in(centers)
        vp  = self.W_out(contexts)
        vn  = self.W_out(negatives)
        loss = vc.new_zeros(1).squeeze()
        for m in self.nesting:
            loss = loss + self.lam * neg_sampling_loss(vc[:,:m], vp[:,:m], vn[:,:,:m])
        return loss


print("Models loaded.")


Models loaded.


In [8]:
# ═══════════════════════════════════════════════════
# CELL 4 — TRAINING
# v2 fixes vs v1 (complete list):
#   ✗ SparseAdam       → silent CPU fallback on CUDA
#   ✗ AMP/GradScaler   → incompatible with sparse grads
#   ✗ torch.compile    → crashes on SparseCUDA backward
#   ✗ clip_grad_value_ → no SparseCUDA kernel
#   ✗ DataLoader       → multiprocessing breaks on Kaggle
#   ✗ DataParallel     → incompatible with sparse embeddings
#   ✓ Dense Adam + CosineAnnealingLR
#   ✓ Full dataset on GPU (eliminates DataLoader bottleneck)
#   ✓ GPU NegativeSampler (zero CPU cost per batch)
#   ✓ DataParallel (works with dense embeddings)
#   ✓ Full checkpoint resume (model + optim + scheduler state)
#   ✓ vocab_size read from checkpoint (fixes size mismatch error)
# ═══════════════════════════════════════════════════

import os, time, pickle
import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm


class NegativeSampler:
    """All sampling on GPU — zero CPU involvement per batch."""
    def __init__(self, vocab, device):
        self.probs  = torch.tensor(vocab.neg_sample_probs()).to(device)
        self.device = device

    @torch.no_grad()
    def sample(self, batch_size):
        return torch.multinomial(
            self.probs,
            num_samples=batch_size * NEG_SAMPLES,
            replacement=True,
        ).view(batch_size, NEG_SAMPLES)


def load_pairs_to_gpu(dataset, device):
    pairs = torch.tensor(dataset.pairs, dtype=torch.long).to(device)
    print(f"  Pairs on GPU: {pairs.shape}  ({pairs.nbytes/1e6:.0f} MB)")
    return pairs


def train(model_type, vocab, dataset, save_path, epochs=EPOCHS):
    """
    Train for `epochs` more epochs, resuming from checkpoint if it exists.
    Calling this repeatedly always continues from where training left off.
    """
    device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
    n_gpus = torch.cuda.device_count() if device.type == "cuda" else 0

    model = StandardWord2Vec(vocab.size) if model_type == "standard" else MRLWord2Vec(vocab.size)

    # ── Load data onto GPU ────────────────────────────────────────────────────
    neg_sampler     = NegativeSampler(vocab, device)
    pairs_gpu       = load_pairs_to_gpu(dataset, device)
    N               = len(pairs_gpu)
    steps_per_epoch = N // BATCH_SIZE
    total_steps     = epochs * steps_per_epoch

    optimizer = Adam(model.parameters(), lr=LR)
    scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=LR*0.01)

    # ── Resume ────────────────────────────────────────────────────────────────
    epochs_done = 0
    if os.path.exists(save_path):
        ckpt        = torch.load(save_path, map_location="cpu", weights_only=False)
        epochs_done = ckpt.get("epochs_trained", 0)
        model.load_state_dict(ckpt["model_state"])
        if "optimizer_state" in ckpt: optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" in ckpt: scheduler.load_state_dict(ckpt["scheduler_state"])
        print(f"  Resumed from epoch {epochs_done}")
    else:
        print(f"  No checkpoint — training from scratch")

    model = model.to(device)
    if n_gpus > 1:
        model = torch.nn.DataParallel(model)
        print(f"  DataParallel: {n_gpus} GPUs")

    print(f"\n{'='*60}")
    print(f"  Training : {model_type.upper()}")
    print(f"  Device   : {device}  |  GPUs: {n_gpus}")
    print(f"  Epochs   : {epochs_done+1} → {epochs_done+epochs}")
    print(f"  Batch    : {BATCH_SIZE} ({BATCH_SIZE//max(n_gpus,1)} per GPU)")
    print(f"{'='*60}")

    os.makedirs(SAVE_DIR, exist_ok=True)
    step = 0
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss, t0 = 0.0, time.time()
        perm      = torch.randperm(N, device=device)
        pairs_gpu = pairs_gpu[perm]

        pbar = tqdm(range(steps_per_epoch),
                    desc=f"Epoch {epochs_done+epoch}/{epochs_done+epochs}",
                    dynamic_ncols=True)
        for i in pbar:
            batch     = pairs_gpu[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
            centers   = batch[:, 0]
            contexts  = batch[:, 1]
            negatives = neg_sampler.sample(len(centers))

            optimizer.zero_grad(set_to_none=True)
            loss = model(centers, contexts, negatives)
            if loss.dim() > 0: loss = loss.mean()
            loss.backward()
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            step       += 1
            if i % 100 == 0:
                pbar.set_postfix(loss=f"{epoch_loss/(i+1):.4f}",
                                 lr=f"{scheduler.get_last_lr()[0]:.2e}")

        elapsed = time.time() - t0
        print(f"  Epoch {epochs_done+epoch} | loss {epoch_loss/steps_per_epoch:.4f} | "
              f"{elapsed:.0f}s | {steps_per_epoch*BATCH_SIZE/elapsed/1e6:.2f}M pairs/s")

    # ── Save full state ────────────────────────────────────────────────────────
    raw = getattr(model, "_orig_mod", model)
    raw = getattr(raw,   "module",    raw)
    torch.save({
        "model_state":     raw.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "model_type":      model_type,
        "vocab_size":      vocab.size,
        "epochs_trained":  epochs_done + epochs,
    }, save_path)
    print(f"  Saved → {save_path}  (total: {epochs_done+epochs} epochs)\n")
    return model


def load_model(save_path):
    """
    v2 fix: reads vocab_size FROM checkpoint — no hardcoded size.
    v1 crashed with size mismatch when vocab_size=50000 was passed
    but checkpoint had 47134 (actual vocab after MIN_COUNT filtering).
    """
    ckpt       = torch.load(save_path, map_location="cpu", weights_only=False)
    vocab_size = ckpt["vocab_size"]          # ← always read from checkpoint
    mtype      = ckpt["model_type"]
    model      = StandardWord2Vec(vocab_size) if mtype == "standard" else MRLWord2Vec(vocab_size)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"  Loaded {mtype} | vocab={vocab_size} | "
          f"epochs={ckpt.get('epochs_trained','?')}")
    return model


print("Training module loaded.")


Training module loaded.


In [9]:
# ═══════════════════════════════════════════════════
# CELL 5 — PHASE TRAIN
# Run this cell to train / continue training.
# Automatically resumes from checkpoint if it exists.
# ═══════════════════════════════════════════════════

def phase_train():
    download_corpus()
    tokens = load_tokens()
    vocab  = Vocabulary(tokens)
    print(f"Vocabulary size: {vocab.size:,}")

    os.makedirs(SAVE_DIR, exist_ok=True)
    with open(f"{SAVE_DIR}/vocab.pkl", "wb") as f:
        pickle.dump(vocab, f)

    dataset = SkipGramDataset(vocab, tokens)
    train("standard", vocab, dataset, STD_CKPT)
    train("mrl",      vocab, dataset, MRL_CKPT)
    print("\n✓ Training complete.")

phase_train()


Corpus ready: data/text8
Vocabulary size: 47,134
Loading cached pairs from data/skipgram_pairs.npy …
Dataset: 53,955,910 pairs  (432 MB, int32)
  Pairs on GPU: torch.Size([53955910, 2])  (863 MB)
  No checkpoint — training from scratch
  DataParallel: 2 GPUs

  Training : STANDARD
  Device   : cuda  |  GPUs: 2
  Epochs   : 1 → 10
  Batch    : 65536 (32768 per GPU)


Epoch 1/10:   0%|          | 0/823 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Epoch 1/10: 100%|██████████| 823/823 [00:42<00:00, 19.23it/s, loss=3.5127, lr=9.77e-04]


  Epoch 1 | loss 3.5055 | 43s | 1.25M pairs/s


Epoch 2/10: 100%|██████████| 823/823 [00:41<00:00, 19.95it/s, loss=3.0565, lr=9.08e-04]


  Epoch 2 | loss 3.0519 | 41s | 1.31M pairs/s


...

  Epoch 8 | loss 2.7802 | 105s | 0.51M pairs/s


Epoch 9/10: 100%|██████████| 823/823 [01:45<00:00,  7.82it/s, loss=2.7744, lr=3.55e-05]


  Epoch 9 | loss 2.7743 | 105s | 0.51M pairs/s


Epoch 10/10: 100%|██████████| 823/823 [01:45<00:00,  7.81it/s, loss=2.7719, lr=1.00e-05]


  Epoch 10 | loss 2.7719 | 105s | 0.51M pairs/s
  Saved → checkpoints/mrl_w2v.pt  (total: 10 epochs)


✓ Training complete.


In [19]:
# ═══════════════════════════════════════════════════
# CELL 15 — EXPORT
# Zip embeddings and checkpoints for download.
# ═══════════════════════════════════════════════════

import zipfile, pickle
from IPython.display import FileLink, display

with open(f"{SAVE_DIR}/vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

std_model = load_model(STD_CKPT)
mrl_model = load_model(MRL_CKPT)
std_emb   = std_model.get_embeddings()
mrl_emb   = mrl_model.get_embeddings()

os.makedirs("exports", exist_ok=True)
np.save("exports/standard_embeddings.npy", std_emb)
np.save("exports/mrl_embeddings.npy",      mrl_emb)
np.save("exports/vocab_words.npy",         np.array(vocab.i2w))

print(f"Standard : {std_emb.shape}  {std_emb.nbytes/1e6:.1f} MB")
print(f"MRL      : {mrl_emb.shape}  {mrl_emb.nbytes/1e6:.1f} MB")
print(f"Vocab    : {len(vocab.i2w)} words")

# ── Embeddings zip ────────────────────────────────
with zipfile.ZipFile("mrl_bias_v2_embeddings.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("exports/standard_embeddings.npy", "standard_embeddings.npy")
    zf.write("exports/mrl_embeddings.npy",       "mrl_embeddings.npy")
    zf.write("exports/vocab_words.npy",          "vocab_words.npy")
    zf.write(f"{SAVE_DIR}/vocab.pkl",            "vocab.pkl")
sz = os.path.getsize("mrl_bias_v2_embeddings.zip")/1e6
print(f"\nEmbeddings zip: {sz:.1f} MB")

# ── Checkpoints zip ───────────────────────────────
with zipfile.ZipFile("mrl_bias_v2_checkpoints.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(SAVE_DIR):
        fp = os.path.join(SAVE_DIR, fn)
        zf.write(fp, fn)
        print(f"  Added checkpoint: {fn}  ({os.path.getsize(fp)/1e6:.1f} MB)")
sz = os.path.getsize("mrl_bias_v2_checkpoints.zip")/1e6
print(f"Checkpoints zip: {sz:.1f} MB")

# ── Download links ────────────────────────────────
display(FileLink("mrl_bias_v2_embeddings.zip"))
display(FileLink("mrl_bias_v2_checkpoints.zip"))


  Loaded standard | vocab=47134 | epochs=10
  Loaded mrl | vocab=47134 | epochs=10
Standard : (47134, 300)  56.6 MB
MRL      : (47134, 300)  56.6 MB
Vocab    : 47134 words

Embeddings zip: 105.5 MB
  Added checkpoint: standard_w2v.pt  (339.4 MB)
  Added checkpoint: mrl_w2v.pt  (339.4 MB)
  Added checkpoint: vocab.pkl  (1.2 MB)
Checkpoints zip: 622.7 MB


/kaggle/working/mrl_bias_v2_embeddings.zip

/kaggle/working/mrl_bias_v2_checkpoints.zip